# Multi-Class Classification - Handwritten Digit Recognition
## Module 2, Session 4: Classifying 10 Digit Classes (0-9)

**Goal:** Build a classifier that recognizes handwritten digits  
**Dataset:** 8x8 pixel digit images (sklearn digits dataset)  
**Problem:** Multi-class classification (10 classes: 0-9)  
**Target Accuracy:** >90%  

---

### What You'll Learn
1. Multi-class classification (more than 2 classes)
2. Working with image data
3. Confusion matrix for 10 classes
4. Per-class precision and recall
5. Identifying which digits get confused
6. Support Vector Machine (SVM) classifier

---

**Created:** 2026-01-06  
**Course:** ML for Engineers  
**Module:** 2 - Classification

## Step 1: Import Libraries

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Dataset
from sklearn.datasets import load_digits

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Machine Learning
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

# Metrics
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
np.random.seed(42)

print("✓ All libraries imported successfully!")

## Step 2: Load the Digits Dataset

We'll use the sklearn digits dataset - 1,797 8x8 pixel images of handwritten digits.

> **AI Assistant Prompt:**  
> "Load the sklearn digits dataset and explain what it contains. Show how images are represented as 8x8 pixel arrays with values 0-16 representing grayscale intensity."

In [ ]:
# Load digits dataset
digits = load_digits()

print("✓ Digits dataset loaded!")
print(f"\nDataset details:")
print(f"  Total images: {len(digits.data)}")
print(f"  Image size: {digits.images[0].shape} (8x8 pixels)")
print(f"  Features per image: {len(digits.data[0])} (8x8 = 64 pixels)")
print(f"  Number of classes: {len(digits.target_names)}")
print(f"  Classes: {digits.target_names}")

print("\n📊 Each image is an 8x8 grid of grayscale pixels.")
print("   Pixel values range from 0 (white) to 16 (black).")
print("   These 64 pixels become our 64 features!")

In [ ]:
# Visualize sample digits
fig, axes = plt.subplots(2, 5, figsize=(12, 6))
fig.suptitle('Sample Handwritten Digits', fontsize=16, fontweight='bold')

for i, ax in enumerate(axes.flat):
    ax.imshow(digits.images[i], cmap='gray')
    ax.set_title(f'Label: {digits.target[i]}', fontsize=12, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

print("\n👀 These are actual handwritten digits from the dataset!")
print("   Notice how each digit looks different - some are neat, some messy.")
print("   Our model needs to learn what makes each digit unique.")

In [ ]:
# Show how an image becomes features
sample_image = digits.images[0]
sample_features = digits.data[0]

print("How an image becomes features:")
print("\n8x8 Image (pixel intensities):")
print(sample_image)

print("\nFlattened to 64 features:")
print(sample_features)

print(f"\nThis image is a '{digits.target[0]}'")
print("\n💡 The ML model sees this digit as 64 numbers!")

## Step 3: Exploratory Data Analysis

In [ ]:
# Create DataFrame
df = pd.DataFrame(digits.data)
df['target'] = digits.target

print(f"Dataset shape: {df.shape}")
print(f"Features: {df.shape[1] - 1}")
print(f"Samples: {df.shape[0]}")

# Class distribution
print("\nClass Distribution:")
print(df['target'].value_counts().sort_index())

In [ ]:
# Visualize class distribution
plt.figure(figsize=(12, 5))

counts = df['target'].value_counts().sort_index()
plt.bar(counts.index, counts.values, color='#45B7D1', alpha=0.8, edgecolor='black')

# Add value labels
for i, v in enumerate(counts.values):
    plt.text(i, v, str(v), ha='center', va='bottom', fontweight='bold')

plt.xlabel('Digit Class', fontsize=12, fontweight='bold')
plt.ylabel('Number of Samples', fontsize=12, fontweight='bold')
plt.title('Distribution of Digit Classes', fontsize=14, fontweight='bold')
plt.xticks(range(10))
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✓ Fairly balanced dataset - each digit has roughly the same number of samples!")

In [ ]:
# Show multiple examples of each digit
fig, axes = plt.subplots(10, 5, figsize=(10, 18))
fig.suptitle('5 Examples of Each Digit (0-9)', fontsize=16, fontweight='bold')

for digit in range(10):
    # Get indices of this digit
    digit_indices = np.where(digits.target == digit)[0]
    # Select 5 random examples
    samples = np.random.choice(digit_indices, 5, replace=False)
    
    for i, sample_idx in enumerate(samples):
        axes[digit, i].imshow(digits.images[sample_idx], cmap='gray')
        axes[digit, i].axis('off')
        if i == 0:
            axes[digit, i].set_ylabel(f'Digit {digit}', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n💡 Notice how the same digit can look very different!")
print("   Handwriting varies - that's what makes this challenging.")

## Step 4: Prepare Data for Training

In [ ]:
# Separate features and target
X = digits.data
y = digits.target

print(f"Features (X): {X.shape}")
print(f"Target (y): {y.shape}")

print(f"\nFeature range: {X.min():.0f} to {X.max():.0f}")
print(f"Target classes: {np.unique(y)}")

In [ ]:
# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # Maintain class distribution
)

print("✓ Data split complete!\n")
print(f"Training samples: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Testing samples: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")

print(f"\nTraining set class distribution:")
unique, counts = np.unique(y_train, return_counts=True)
for digit, count in zip(unique, counts):
    print(f"  Digit {digit}: {count} samples")

## Step 5: Train Support Vector Machine (SVM) Classifier

**SVM** is excellent for image classification!

> **AI Assistant Prompt:**  
> "Train a Support Vector Machine (SVM) classifier with RBF kernel on the digits dataset. Explain why SVM works well for multi-class image classification."

In [ ]:
print("\n" + "="*60)
print("TRAINING SUPPORT VECTOR MACHINE (SVM)")
print("="*60)

# Create SVM model
svm_model = SVC(
    kernel='rbf',       # Radial Basis Function kernel
    gamma='scale',      # Kernel coefficient
    C=10,               # Regularization parameter
    random_state=42
)

print("\n[1/2] Training SVM classifier...")
svm_model.fit(X_train, y_train)

print("[2/2] Making predictions...")
svm_predictions = svm_model.predict(X_test)

# Calculate accuracy
svm_accuracy = accuracy_score(y_test, svm_predictions)

print("\n✓ TRAINING COMPLETE!")
print("="*60)
print(f"\n🎯 SVM Accuracy: {svm_accuracy*100:.2f}%")

if svm_accuracy >= 0.95:
    print("\n🎉 OUTSTANDING! Exceptional performance!")
elif svm_accuracy >= 0.90:
    print("\n🎉 EXCELLENT! Target accuracy achieved!")
elif svm_accuracy >= 0.85:
    print("\n👍 GOOD! Close to target.")
else:
    print("\n⚠️ Model could be better. Try tuning hyperparameters.")

## Step 6: Detailed Performance Analysis

For multi-class problems, we need to analyze performance for EACH class!

In [ ]:
# Detailed classification report
print("Classification Report - Per-Class Metrics:")
print("="*60)
print(classification_report(
    y_test,
    svm_predictions,
    target_names=[f'Digit {i}' for i in range(10)]
))

print("\n📖 Understanding Per-Class Metrics:")
print("  • Precision: When we predict digit X, how often are we right?")
print("  • Recall: Of all actual digit X images, how many did we find?")
print("  • F1-score: Harmonic mean of precision and recall")
print("  • Support: Number of actual samples for each digit in test set")

In [ ]:
# Visualize per-class performance
report = classification_report(
    y_test, 
    svm_predictions, 
    output_dict=True,
    target_names=[f'Digit {i}' for i in range(10)]
)

# Extract per-class metrics
digits_list = [f'Digit {i}' for i in range(10)]
precision = [report[d]['precision'] for d in digits_list]
recall = [report[d]['recall'] for d in digits_list]
f1 = [report[d]['f1-score'] for d in digits_list]

# Plot
fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(10)
width = 0.25

ax.bar(x - width, precision, width, label='Precision', color='#FF6B6B', alpha=0.8)
ax.bar(x, recall, width, label='Recall', color='#4ECDC4', alpha=0.8)
ax.bar(x + width, f1, width, label='F1-Score', color='#45B7D1', alpha=0.8)

ax.set_xlabel('Digit Class', fontsize=12, fontweight='bold')
ax.set_ylabel('Score', fontsize=12, fontweight='bold')
ax.set_title('Per-Class Performance Metrics', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(range(10))
ax.legend()
ax.set_ylim([0.7, 1.05])
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n💡 Look for digits with lower scores - those are harder to classify!")

## Step 7: Confusion Matrix for 10 Classes

The confusion matrix shows which digits get confused with each other.

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(y_test, svm_predictions)

# Visualize with larger size for 10 classes
plt.figure(figsize=(12, 10))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=range(10)
)
disp.plot(cmap='Blues', ax=plt.gca(), colorbar=True)
plt.title('Confusion Matrix - 10 Digit Classes', fontsize=16, fontweight='bold')
plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
plt.ylabel('True Label', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n📊 How to Read This Confusion Matrix:")
print("  • Rows = Actual digit")
print("  • Columns = Predicted digit")
print("  • Diagonal = Correct predictions (bright blue)")
print("  • Off-diagonal = Misclassifications")
print("\n  Example: If row 8, column 3 has value 2:")
print("  → 2 actual '8' digits were wrongly classified as '3'")

In [ ]:
# Find most confused digit pairs
print("\nMost Common Misclassifications:")
print("="*60)

# Create a copy and zero out the diagonal
cm_errors = cm.copy()
np.fill_diagonal(cm_errors, 0)

# Find top misclassifications
top_errors = []
for true_label in range(10):
    for pred_label in range(10):
        if cm_errors[true_label, pred_label] > 0:
            top_errors.append({
                'True': true_label,
                'Predicted': pred_label,
                'Count': cm_errors[true_label, pred_label]
            })

# Sort by count
top_errors = sorted(top_errors, key=lambda x: x['Count'], reverse=True)[:10]

if top_errors:
    for i, error in enumerate(top_errors, 1):
        print(f"{i}. Digit '{error['True']}' misclassified as '{error['Predicted']}': {error['Count']} times")
    
    print("\n💡 These digit pairs look similar to the model!")
    print("   Common confusions: 8↔3, 5↔3, 9↔4, etc.")
else:
    print("✓ Perfect classification! No errors!")

In [ ]:
# Show examples of misclassified digits
misclassified_indices = np.where(y_test != svm_predictions)[0]

if len(misclassified_indices) > 0:
    print(f"\nTotal misclassifications: {len(misclassified_indices)}")
    print(f"Error rate: {len(misclassified_indices)/len(y_test)*100:.2f}%")
    
    # Show first 10 errors
    n_show = min(10, len(misclassified_indices))
    fig, axes = plt.subplots(2, 5, figsize=(12, 6))
    fig.suptitle('Misclassified Examples', fontsize=16, fontweight='bold', color='red')
    
    for i, ax in enumerate(axes.flat):
        if i < n_show:
            idx = misclassified_indices[i]
            # Get the test index in original dataset
            test_idx = np.where((digits.data == X_test[idx]).all(axis=1))[0][0]
            
            ax.imshow(digits.images[test_idx], cmap='gray')
            ax.set_title(
                f'True: {y_test[idx]}\nPred: {svm_predictions[idx]}',
                fontsize=10,
                fontweight='bold',
                color='red'
            )
            ax.axis('off')
        else:
            ax.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print("\n👀 Look at these errors - can you see why the model got confused?")
else:
    print("\n🎉 PERFECT! No misclassifications!")

## Step 8: Compare Multiple Algorithms

In [ ]:
print("\n" + "="*60)
print("COMPARING MULTIPLE ALGORITHMS")
print("="*60)

# Define models
models = {
    'SVM': SVC(kernel='rbf', C=10, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42)
}

# Train and evaluate
results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    results[name] = accuracy
    print(f"  ✓ Accuracy: {accuracy*100:.2f}%")

print("\n" + "="*60)

In [ ]:
# Visualize comparison
plt.figure(figsize=(12, 6))
models_list = list(results.keys())
accuracies = [results[m]*100 for m in models_list]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']

bars = plt.bar(models_list, accuracies, color=colors, alpha=0.8, edgecolor='black')

# Add value labels
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}%',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

# Add target line
plt.axhline(y=90, color='red', linestyle='--', linewidth=2, label='Target (90%)')

plt.title('Algorithm Comparison - Digit Recognition', fontsize=14, fontweight='bold')
plt.xlabel('Algorithm', fontsize=12)
plt.ylabel('Accuracy (%)', fontsize=12)
plt.ylim([80, 105])
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

# Best model
best_model = max(results, key=results.get)
best_accuracy = results[best_model]

print(f"\n🏆 BEST MODEL: {best_model}")
print(f"   Accuracy: {best_accuracy*100:.2f}%")

## Step 9: Test with New Handwritten Digits

In [ ]:
# Predict on random test samples
print("\n" + "="*70)
print("TESTING DIGIT RECOGNITION")
print("="*70)

# Select 12 random test samples
n_samples = 12
sample_indices = np.random.choice(len(X_test), n_samples, replace=False)

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
fig.suptitle('Digit Recognition Predictions', fontsize=16, fontweight='bold')

for i, ax in enumerate(axes.flat):
    idx = sample_indices[i]
    
    # Get the test sample
    sample = X_test[idx:idx+1]
    true_label = y_test[idx]
    
    # Predict
    prediction = svm_model.predict(sample)[0]
    
    # Get image
    test_idx = np.where((digits.data == X_test[idx]).all(axis=1))[0][0]
    
    # Display
    ax.imshow(digits.images[test_idx], cmap='gray')
    
    if prediction == true_label:
        color = 'green'
        symbol = '✓'
    else:
        color = 'red'
        symbol = '✗'
    
    ax.set_title(
        f'{symbol} Pred: {prediction} | True: {true_label}',
        fontsize=10,
        fontweight='bold',
        color=color
    )
    ax.axis('off')

plt.tight_layout()
plt.show()

# Calculate accuracy on these samples
sample_predictions = svm_model.predict(X_test[sample_indices])
sample_accuracy = accuracy_score(y_test[sample_indices], sample_predictions)

print(f"\nAccuracy on these {n_samples} samples: {sample_accuracy*100:.1f}%")
print("\n✓ Your digit recognizer is working!")

## Step 10: Summary and Key Takeaways

In [ ]:
print("\n" + "="*70)
print("PROJECT SUMMARY - MULTI-CLASS DIGIT CLASSIFICATION")
print("="*70)

print("\n✅ WHAT YOU ACCOMPLISHED:")
print("  1. Loaded and explored digit images dataset")
print("  2. Visualized handwritten digits (8x8 pixels)")
print("  3. Understood how images become features")
print("  4. Trained SVM classifier for 10 classes")
print(f"  5. Achieved {svm_accuracy*100:.2f}% accuracy")
print("  6. Analyzed per-class performance metrics")
print("  7. Created confusion matrix for 10 classes")
print("  8. Identified which digits get confused")
print("  9. Compared 4 different algorithms")

print("\n🎯 KEY LEARNINGS:")
print("  • Multi-class classification handles 3+ classes")
print("  • Images can be flattened into feature vectors")
print("  • SVM works exceptionally well for image data")
print("  • Confusion matrix reveals which classes are similar")
print("  • Per-class metrics show performance for each digit")
print("  • Some digits (like 8 and 3) are easier to confuse")

print("\n📊 FINAL RESULTS:")
print(f"  • Dataset: {len(digits.data)} digit images")
print(f"  • Image size: 8x8 pixels = 64 features")
print(f"  • Number of classes: 10 (digits 0-9)")
print(f"  • Training samples: {len(X_train)}")
print(f"  • Testing samples: {len(X_test)}")
print(f"  • Best model: {best_model}")
print(f"  • Best accuracy: {best_accuracy*100:.2f}%")
print(f"  • Target achieved: {'✅ YES' if best_accuracy >= 0.90 else '⚠️ CLOSE'}")
print(f"  • Misclassifications: {len(misclassified_indices) if len(misclassified_indices) > 0 else 0}")

print("\n🔍 INTERESTING FINDINGS:")
if top_errors:
    print(f"  • Most confused pair: {top_errors[0]['True']} ↔ {top_errors[0]['Predicted']}")
    print(f"  • These digits look visually similar!")
else:
    print("  • Perfect classification - no confusion!")

print("\n🚀 NEXT STEPS:")
print("  • Session 5: Feature engineering techniques")
print("  • Learn to create new features from existing ones")
print("  • Understand feature scaling and normalization")
print("  • Compare model performance before/after engineering")

print("\n💡 REAL-WORLD APPLICATIONS:")
print("  • Handwriting recognition (postal codes, checks)")
print("  • Document digitization (OCR)")
print("  • Form processing automation")
print("  • License plate recognition")
print("  • Historical document preservation")

print("\n" + "="*70)
print("🎉 YOU MASTERED MULTI-CLASS CLASSIFICATION!")
print("="*70)

---

## AI Prompts for This Notebook

### Prompt 1: Dataset Loading
```
Load the sklearn digits dataset and explain what it contains. Show how 8x8 pixel 
images are represented as arrays and visualize 10 sample digits with their labels.
```

### Prompt 2: Multi-Class Visualization
```
For the digits dataset, create a grid showing 5 examples of each digit (0-9). 
This should be a 10x5 grid where each row is a different digit class. Show the 
variety in handwriting styles.
```

### Prompt 3: SVM Training
```
Train a Support Vector Machine (SVM) with RBF kernel on the digits dataset. 
Split 80/20 train/test. Calculate accuracy and show a detailed classification 
report with per-class metrics.
```

### Prompt 4: Confusion Matrix
```
Create a 10x10 confusion matrix for digit classification. Make it large and 
readable. Then find and list the top 10 most common misclassification pairs 
(e.g., '8' classified as '3').
```

### Prompt 5: Per-Class Metrics
```
Extract precision, recall, and F1-score for each of the 10 digit classes. 
Create a grouped bar chart showing all three metrics side-by-side for each digit. 
Which digits are hardest to classify?
```

### Prompt 6: Error Analysis
```
Find all misclassified digits in the test set. Display the first 10 
misclassifications showing both the true label and predicted label. Can you see 
why the model got confused?
```

---

**End of Notebook**

**Created:** 2026-01-06  
**Course:** ML for Engineers - Module 2  
**Project:** Multi-Class Digit Recognition  
**Target:** >90% Accuracy ✅